# 02 — LTV Prediction

## Business Problem

A digital wallet company wants to predict customer lifetime value so that marketing, retention, and support teams can make better decisions.

Business questions:

1. Which customers are expected to generate high value?
2. Which features are predictive of LTV?
3. How much acquisition cost can we afford by segment?
4. Which users are profitable enough to receive retention incentives?
5. How do we avoid target leakage when predicting LTV?

## Senior-Level Framing

This notebook compares:

- **Snapshot model:** uses most available customer-level features
- **Leakage-aware model:** excludes variables that may contain future/lifetime information

This is important because many LTV datasets contain accumulated variables that make the model look unrealistically strong.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RETENTION_PATH = PROCESSED_DIR / "wallet_retention_baseline.csv"
RAW_PATH = RAW_DIR / "digital_wallet_ltv_dataset.csv"

if RETENTION_PATH.exists():
    df = pd.read_csv(RETENTION_PATH)
    print("Loaded processed retention baseline:", RETENTION_PATH)
elif RAW_PATH.exists():
    df = pd.read_csv(RAW_PATH)
    print("Loaded raw wallet dataset:", RAW_PATH)
else:
    raise FileNotFoundError("Place digital_wallet_ltv_dataset.csv under data/raw/ or run 01 first.")

print("Shape:", df.shape)
df.head()

## 2. Target and Schema Review

In [ ]:
TARGET = "LTV"
ID_COL = "Customer_ID"

display(df.head())
display(df.describe(include="all").T)

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

In [ ]:
plt.figure(figsize=(8, 4))
df[TARGET].hist(bins=40)
plt.title("LTV Distribution")
plt.xlabel("LTV")
plt.ylabel("Customer count")
plt.show()

plt.figure(figsize=(8, 4))
np.log1p(df[TARGET]).hist(bins=40)
plt.title("log1p(LTV) Distribution")
plt.xlabel("log1p(LTV)")
plt.ylabel("Customer count")
plt.show()

target_summary = pd.DataFrame({
    "metric": ["min", "p25", "median", "mean", "p75", "p90", "p95", "p99", "max", "skew"],
    "value": [
        df[TARGET].min(),
        df[TARGET].quantile(0.25),
        df[TARGET].median(),
        df[TARGET].mean(),
        df[TARGET].quantile(0.75),
        df[TARGET].quantile(0.90),
        df[TARGET].quantile(0.95),
        df[TARGET].quantile(0.99),
        df[TARGET].max(),
        df[TARGET].skew(),
    ]
})
display(target_summary)

## 3. Leakage Risk Analysis

Potential leakage-like features:

- `Total_Spent`: may be directly part of LTV definition
- `Loyalty_Points_Earned`: may accumulate with spending
- `Cashback_Received`: may be related to historical spend
- `monetary_score`: derived from `Total_Spent`
- `rfm_score`: includes monetary score
- `ltv_quartile`, `high_value`, `high_value_at_risk`: directly or partly derived from LTV

We will compare a snapshot model and a leakage-aware model.

In [ ]:
corr = df.select_dtypes(include=["number"]).corr(numeric_only=True)

if TARGET in corr.columns:
    ltv_corr = (
        corr[TARGET]
        .drop(TARGET)
        .sort_values(key=abs, ascending=False)
        .to_frame("correlation_with_LTV")
    )
    display(ltv_corr)

    plt.figure(figsize=(8, 5))
    ltv_corr.head(15).sort_values("correlation_with_LTV").plot(kind="barh", legend=False)
    plt.title("Top Numeric Correlations with LTV")
    plt.xlabel("Correlation")
    plt.show()

direct_target_leakage = ["ltv_quartile", "high_value", "high_value_at_risk"]
possible_time_leakage = ["Total_Spent", "Loyalty_Points_Earned", "Cashback_Received", "monetary_score", "rfm_score"]

print("Direct leakage present:", [c for c in direct_target_leakage if c in df.columns])
print("Possible time leakage present:", [c for c in possible_time_leakage if c in df.columns])

## 4. Feature Set Design

In [ ]:
base_exclude = [ID_COL, TARGET]

direct_leakage_cols = ["ltv_quartile", "high_value", "high_value_at_risk"]
snapshot_exclude = base_exclude + direct_leakage_cols

snapshot_features = [c for c in df.columns if c not in snapshot_exclude]

leakage_aware_exclude = snapshot_exclude + [
    "Total_Spent",
    "Loyalty_Points_Earned",
    "Cashback_Received",
    "monetary_score",
    "rfm_score",
    "rfm_segment",
]

leakage_aware_features = [c for c in df.columns if c not in leakage_aware_exclude]

print("Snapshot feature count:", len(snapshot_features))
print(snapshot_features)

print("\nLeakage-aware feature count:", len(leakage_aware_features))
print(leakage_aware_features)

## 5. Modeling Utilities

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

def top_quartile_capture(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    true_top = y_true >= np.quantile(y_true, 0.75)
    pred_top = y_pred >= np.quantile(y_pred, 0.75)
    return (true_top & pred_top).sum() / true_top.sum()

def make_preprocessor(X):
    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ])

    return preprocessor

## 6. Train Baseline and ML Models

In [ ]:
def run_ltv_experiment(data, feature_cols, experiment_name, log_target=False):
    X = data[feature_cols].copy()
    y_original = data[TARGET].copy()
    y_model = np.log1p(y_original) if log_target else y_original

    X_train, X_test, y_train, y_test, y_train_orig, y_test_orig = train_test_split(
        X, y_model, y_original, test_size=0.25, random_state=RANDOM_STATE
    )

    preprocessor = make_preprocessor(X_train)

    models = {
        "mean_baseline": None,
        "linear_regression": LinearRegression(),
        "ridge": Ridge(alpha=10.0),
        "random_forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "gradient_boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    }

    rows = []
    fitted_models = {}

    for model_name, model in models.items():
        if model is None:
            pred_model_scale = np.repeat(y_train.mean(), len(y_test))
            fitted_models[model_name] = None
        else:
            pipe = Pipeline([("preprocess", preprocessor), ("model", model)])
            pipe.fit(X_train, y_train)
            pred_model_scale = pipe.predict(X_test)
            fitted_models[model_name] = pipe

        pred = np.expm1(pred_model_scale) if log_target else pred_model_scale
        pred = np.maximum(pred, 0)

        metrics = regression_metrics(y_test_orig, pred)
        metrics["top_quartile_capture"] = top_quartile_capture(y_test_orig, pred)
        metrics["model"] = model_name
        metrics["experiment"] = experiment_name
        metrics["log_target"] = log_target
        rows.append(metrics)

    split_data = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "y_train_original": y_train_orig,
        "y_test_original": y_test_orig,
        "features": feature_cols,
        "log_target": log_target,
    }

    return pd.DataFrame(rows).sort_values("RMSE"), fitted_models, split_data

snapshot_results, snapshot_models, snapshot_split = run_ltv_experiment(
    df, snapshot_features, "snapshot", log_target=False
)
aware_results, aware_models, aware_split = run_ltv_experiment(
    df, leakage_aware_features, "leakage_aware", log_target=False
)
aware_log_results, aware_log_models, aware_log_split = run_ltv_experiment(
    df, leakage_aware_features, "leakage_aware_log_target", log_target=True
)

all_results = pd.concat([snapshot_results, aware_results, aware_log_results], ignore_index=True)
display(all_results.sort_values(["experiment", "RMSE"]))

## 7. Model Selection

In [ ]:
candidate_results = all_results[
    all_results["experiment"].isin(["leakage_aware", "leakage_aware_log_target"])
].sort_values("RMSE")

display(candidate_results)

best = candidate_results.iloc[0]
best_experiment = best["experiment"]
best_model_name = best["model"]
best_log_target = bool(best["log_target"])

print("Selected experiment:", best_experiment)
print("Selected model:", best_model_name)
print("Log target:", best_log_target)

if best_experiment == "leakage_aware":
    best_models = aware_models
    best_split = aware_split
elif best_experiment == "leakage_aware_log_target":
    best_models = aware_log_models
    best_split = aware_log_split
else:
    raise ValueError("Unexpected selected experiment.")

best_model = best_models[best_model_name]

if best_model is None:
    raise ValueError("Mean baseline selected. Please inspect the data/model setup.")

## 8. Score All Customers

In [ ]:
X_all = df[best_split["features"]].copy()
pred_model_scale = best_model.predict(X_all)

if best_log_target:
    df["predicted_ltv"] = np.expm1(pred_model_scale)
else:
    df["predicted_ltv"] = pred_model_scale

df["predicted_ltv"] = df["predicted_ltv"].clip(lower=0)
df["ltv_prediction_error"] = df["predicted_ltv"] - df[TARGET]

display(df[[ID_COL, TARGET, "predicted_ltv", "ltv_prediction_error"]].head())
display(df["predicted_ltv"].describe().to_frame("predicted_ltv"))

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df[TARGET], df["predicted_ltv"], alpha=0.4)
plt.title("Actual vs Predicted LTV")
plt.xlabel("Actual LTV")
plt.ylabel("Predicted LTV")
plt.show()

plt.figure(figsize=(8, 4))
df["ltv_prediction_error"].hist(bins=40)
plt.title("LTV Prediction Error Distribution")
plt.xlabel("Prediction error")
plt.ylabel("Customer count")
plt.show()

## 9. Segment Predicted LTV

In [ ]:
df["predicted_ltv_segment"] = pd.qcut(
    df["predicted_ltv"].rank(method="first"),
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
)

segment_summary = (
    df.groupby("predicted_ltv_segment", observed=False)
    .agg(
        customers=(ID_COL, "count"),
        avg_predicted_ltv=("predicted_ltv", "mean"),
        avg_actual_ltv=(TARGET, "mean"),
        median_actual_ltv=(TARGET, "median"),
        avg_total_transactions=("Total_Transactions", "mean"),
        avg_active_days=("Active_Days", "mean"),
        avg_satisfaction=("Customer_Satisfaction_Score", "mean"),
        avg_last_transaction_days=("Last_Transaction_Days_Ago", "mean"),
    )
)

display(segment_summary)

In [ ]:
plt.figure(figsize=(8, 4))
segment_summary["avg_predicted_ltv"].plot(kind="bar")
plt.title("Average Predicted LTV by Segment")
plt.ylabel("Average predicted LTV")
plt.xticks(rotation=30)
plt.show()

plt.figure(figsize=(8, 4))
segment_summary["avg_actual_ltv"].plot(kind="bar")
plt.title("Average Actual LTV by Predicted Segment")
plt.ylabel("Average actual LTV")
plt.xticks(rotation=30)
plt.show()

## 10. Feature Importance

In [ ]:
X_test = best_split["X_test"]
y_test_for_perm = best_split["y_test"] if best_log_target else best_split["y_test_original"]

perm = permutation_importance(
    best_model,
    X_test,
    y_test_for_perm,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
)

importance = (
    pd.DataFrame({"feature": X_test.columns, "importance": perm.importances_mean})
    .sort_values("importance", ascending=False)
)

display(importance.head(20))

plt.figure(figsize=(8, 5))
importance.head(15).sort_values("importance").plot(
    kind="barh",
    x="feature",
    y="importance",
    legend=False,
)
plt.title("Permutation Importance — Selected LTV Model")
plt.show()

## 11. CAC Profitability Simulation

In [ ]:
cac_values = [5, 10, 20, 30, 40, 50, 75, 100, 150]

cac_simulation = pd.DataFrame([
    {
        "CAC": cac,
        "profitable_customer_rate": (df["predicted_ltv"] > cac).mean(),
        "avg_expected_profit": (df["predicted_ltv"] - cac).mean(),
        "total_expected_profit": (df["predicted_ltv"] - cac).sum(),
    }
    for cac in cac_values
])

display(cac_simulation)

plt.figure(figsize=(8, 4))
plt.plot(cac_simulation["CAC"], cac_simulation["avg_expected_profit"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("CAC Sensitivity: Average Expected Profit")
plt.xlabel("Customer Acquisition Cost")
plt.ylabel("Average predicted LTV - CAC")
plt.show()

## 12. Retention Targeting Simulation

In [ ]:
if "dormant_90d" not in df.columns:
    df["dormant_90d"] = (df["Last_Transaction_Days_Ago"] > 90).astype(int)

df["retention_risk_score"] = (
    df["Last_Transaction_Days_Ago"] - df["Last_Transaction_Days_Ago"].min()
) / (
    df["Last_Transaction_Days_Ago"].max() - df["Last_Transaction_Days_Ago"].min()
)

df["retention_target_score"] = df["predicted_ltv"] * df["retention_risk_score"]

df["retention_target_decile"] = pd.qcut(
    df["retention_target_score"].rank(method="first"),
    q=10,
    labels=False,
) + 1

incentive_cost = 5.0
assumed_retention_lift_values = [0.02, 0.05, 0.08, 0.12]

rows = []
for lift in assumed_retention_lift_values:
    for min_decile in [10, 9, 8, 7, 6]:
        targeted = df[df["retention_target_decile"] >= min_decile]
        expected_incremental_value = targeted["predicted_ltv"] * lift
        expected_profit = expected_incremental_value - incentive_cost

        rows.append({
            "assumed_retention_lift": lift,
            "target_decile_min": min_decile,
            "customers_targeted": len(targeted),
            "avg_predicted_ltv": targeted["predicted_ltv"].mean(),
            "avg_risk_score": targeted["retention_risk_score"].mean(),
            "total_expected_profit": expected_profit.sum(),
            "profit_per_targeted_customer": expected_profit.mean(),
        })

retention_simulation = pd.DataFrame(rows).sort_values("total_expected_profit", ascending=False)
display(retention_simulation.head(20))

## 13. Business Recommendation Table

In [ ]:
df["risk_segment"] = pd.qcut(
    df["retention_risk_score"].rank(method="first"),
    q=3,
    labels=["Low Risk", "Medium Risk", "High Risk"],
)

action_summary = (
    df.groupby(["predicted_ltv_segment", "risk_segment"], observed=False)
    .agg(
        customers=(ID_COL, "count"),
        avg_predicted_ltv=("predicted_ltv", "mean"),
        avg_actual_ltv=(TARGET, "mean"),
        avg_risk_score=("retention_risk_score", "mean"),
        avg_target_score=("retention_target_score", "mean"),
    )
    .reset_index()
)

def recommend_action(row):
    if row["predicted_ltv_segment"] in ["High", "Very High"] and row["risk_segment"] == "High Risk":
        return "Retention campaign priority"
    if row["predicted_ltv_segment"] in ["High", "Very High"] and row["risk_segment"] != "High Risk":
        return "Protect / nurture"
    if row["predicted_ltv_segment"] in ["Low", "Medium"] and row["risk_segment"] == "High Risk":
        return "Low-cost retention only"
    return "Low priority"

action_summary["recommended_action"] = action_summary.apply(recommend_action, axis=1)
display(action_summary.sort_values(["predicted_ltv_segment", "risk_segment"]))

## 14. Pitfalls and Interpretation

### Pitfall 1: Reporting only model metrics

Business teams need decisions, not just RMSE.

### Pitfall 2: Ignoring prediction timing

If the model uses accumulated lifetime behavior, it may not work for early LTV prediction.

### Pitfall 3: Treating retention simulation as causal

The simulation assumes a lift. It does not estimate lift. True lift requires A/B testing or uplift modeling.

### Pitfall 4: Optimizing for high conversion instead of incremental value

High-LTV customers may return without incentives. Later uplift modeling will address this.

## 15. Save Outputs

In [ ]:
predictions_path = PROCESSED_DIR / "wallet_ltv_predictions.csv"
results_path = PROCESSED_DIR / "ltv_model_results.csv"
importance_path = PROCESSED_DIR / "ltv_feature_importance.csv"

df.to_csv(predictions_path, index=False)
all_results.to_csv(results_path, index=False)
importance.to_csv(importance_path, index=False)

best_summary = best.to_dict()

report = f'''
# 02 LTV Prediction Summary

## Selected model
- Experiment: {best_experiment}
- Model: {best_model_name}
- Log target: {best_log_target}

## Selected model metrics
- MAE: {best_summary["MAE"]:.4f}
- RMSE: {best_summary["RMSE"]:.4f}
- R2: {best_summary["R2"]:.4f}
- Top-quartile capture: {best_summary["top_quartile_capture"]:.4f}

## Modeling decision
The selected model comes from the leakage-aware feature set to avoid overly optimistic performance from accumulated lifetime variables.

## Business outputs
- Predicted LTV for each customer
- Predicted LTV segments
- CAC profitability simulation
- Retention targeting simulation
- Business action table

## Important limitation
The retention targeting simulation is not causal. It assumes retention lift.
Future notebooks will estimate incrementality using A/B testing and uplift modeling.
'''

report_path = REPORTS_DIR / "02_ltv_prediction_summary.md"
report_path.write_text(report)

print("Saved predictions:", predictions_path)
print("Saved model results:", results_path)
print("Saved feature importance:", importance_path)
print("Saved report:", report_path)

## 16. Interview Discussion Questions

1. What is target leakage in LTV prediction?
2. Why compare a snapshot model and a leakage-aware model?
3. Why is MAE often easier to explain to stakeholders than RMSE?
4. What does top-quartile capture measure?
5. How would you use predicted LTV for CAC decisions?
6. Why is retention targeting simulation not causal?
7. What additional data would you need for true incremental retention modeling?